# From stiffness-controlled gels to fixed histology

An additive step. It does not modify the histology or time-lapse pipelines and
does not re-run them: it reads their outputs, re-measures both systems at a
**matched physical scale**, and uses the gel series to infer things about the
histology defects that a fixed section cannot yield on its own.

## Why a matched scale is not optional

| system | packaged sigma | um/px | physical scale |
|---|---|---|---|
| histology | 70 px | 0.1147 | **8.0 um** |
| gel | 40 px | 0.7042 | **28.2 um** |

Defect density falls roughly as `1/sigma^2`, so a 3.5x mismatch in detection
scale is about a **12x** offset in density before any biology enters. Comparing
the existing numbers directly would be measuring the configuration, not the
tissue.

Matching sigma in micrometres fixes the arithmetic, not the ambiguity — nuclei
are packed in tissue and spread on a gel, so matching physical length and
matching cell diameters are different questions. The way out is to compare
**dimensionless** quantities: `rho * xi^2`, defects per correlation area, where
`xi` is measured rather than chosen.

## What the gels let you infer about the histology

1. **The sign of the stiffness response.** Stiffer means more contractile means
   more active means more defects — but steady-state density goes as activity
   over elasticity, and stiffening raises *both*. Which wins is empirical, the
   gel answers it, and histology alone cannot.
2. **An effective stiffness**, by inverting `rho*(E)`. Falsifiable against
   instrumented microindentation on an adjacent section.
3. **A staging readout.** A region whose `rho*` sits above the steady-state band
   of every stiffness has not equilibrated — evidence the lesion is still
   coarsening rather than mature.
4. **Motility from a static image**, via the gel's `rho*`-to-speed relation.

**Not** licensed: absolute lesion age, absolute in vivo activity, or anything
three-dimensional.

## 1. Install and mount

In [ ]:
!pip install -q git+https://github.com/Danpc11/lung-nematic.git

from google.colab import drive
drive.mount('/content/drive')

## 2. Paths and calibration

In [ ]:
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')

HISTOLOGY_IMAGES = Path('/scratch/histology/data')   # or a Drive path
GEL_FRAME_DIRS = {
    5.0:  DRIVE / 'nhlf_timelapse' / '5kPa',
    10.0: DRIVE / 'nhlf_timelapse' / '10kPa',
    23.0: DRIVE / 'nhlf_timelapse' / '23kPa',
}
TIMELAPSE_RESULTS = DRIVE / 'nhlf_timelapse' / 'resultados'
OUTPUT_DIR = DRIVE / 'crossmap'

HISTOLOGY_MPP = 0.114679
GEL_MPP = 1 / 1.42            # 0.70423 um/px

# The sweep is specified in MICROMETRES and converted per system. Never reuse a
# pixel sigma across systems - that is the mismatch this notebook exists to fix.
SIGMAS_UM = (8.0, 12.0, 18.0, 28.0, 40.0, 60.0)

N_HISTOLOGY_IMAGES = 12       # per diagnosis, for the sweep; None uses all
N_GEL_FRAMES = 5              # steady-state frames per stiffness

from lung_nematic.crossmap import sigmas_for_microns

print(f"{'um':>6} {'histology px':>14} {'gel px':>10}")
for s in SIGMAS_UM:
    h = sigmas_for_microns((s,), HISTOLOGY_MPP)[0]
    g = sigmas_for_microns((s,), GEL_MPP)[0]
    print(f"{s:>6.0f} {h:>14.1f} {g:>10.1f}")

## 3. Is the gel at steady state?

This gate decides whether the comparison is legal at all. A histological lesion
has persisted for months, so the only thing it can be compared against is a
**steady-state** level, not an arbitrary point on a transient. If a series never
plateaus, that stiffness has no temporal anchor and must be excluded.

Steadiness is judged on the births-to-deaths balance rather than a fitted decay
exponent: over a short recording a plateau and a slow power law look alike,
while their nucleation balance does not.

In [ ]:
import pandas as pd
from lung_nematic.crossmap import steady_state_window

kinetics = pd.read_csv(TIMELAPSE_RESULTS / 'defect_kinetics.tsv', sep='\t')

steady = {}
for stiffness, group in kinetics.groupby('stiffness_kPa'):
    verdict = steady_state_window(group.sort_values('frame'))
    steady[stiffness] = verdict
    flag = 'STEADY' if verdict['reached_steady_state'] else 'NOT STEADY'
    print(f"{stiffness:5g} kPa  {flag:11s}  "
          f"mean {verdict.get('mean_n_defects', float('nan')):6.1f} defects  "
          f"{verdict.get('reason', '')}")

usable = [s for s, v in steady.items() if v['reached_steady_state']]
print(f"\nusable for calibration: {usable}")
if len(usable) < 2:
    print("! fewer than two steady stiffnesses - no calibration curve is "
          "possible, and any inferred stiffness below would be fabricated")

## 4. Matched-scale sweep on both systems

Segmentation runs once per image and only the smoothing scale varies, so a
change in the curve means the texture changed rather than the detector.

Comparing whole `rho(sigma)` curves beats comparing single numbers: the scale at
which density collapses *is* the correlation length, so the curve carries its
own calibration.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

from lung_nematic.config import load_default_config
from lung_nematic.io_utils import read_rgb
from lung_nematic.crossmap import gel_scale_sweep, histology_scale_sweep

config = load_default_config()
SUPPORTED = {'.png', '.tif', '.tiff', '.jpg', '.jpeg'}

# --- histology -------------------------------------------------------------
histology_paths = sorted(p for p in Path(HISTOLOGY_IMAGES).rglob('*')
                         if p.suffix.lower() in SUPPORTED)
if N_HISTOLOGY_IMAGES:
    by_dx = {}
    for path in histology_paths:
        by_dx.setdefault(path.name.split('_')[0], []).append(path)
    histology_paths = [p for group in by_dx.values()
                       for p in group[:N_HISTOLOGY_IMAGES]]

histology_rows = []
for path in tqdm(histology_paths, desc='histology'):
    try:
        table = histology_scale_sweep(read_rgb(path), config, SIGMAS_UM,
                                      HISTOLOGY_MPP)
    except ValueError as error:
        print(f"  skipped {path.name}: {error}")
        continue
    table['image_id'] = path.stem
    table['dx'] = path.stem.split('_')[0]
    table['patient'] = path.stem.split('_')[1] if '_' in path.stem else None
    histology_rows.append(table)
histology_sweep = pd.concat(histology_rows, ignore_index=True)

# --- gels, steady-state frames only ---------------------------------------
gel_rows = []
for stiffness in usable:
    folder = GEL_FRAME_DIRS[stiffness]
    paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in SUPPORTED)
    window = steady[stiffness]
    paths = paths[window['first_frame']:window['last_frame'] + 1][:N_GEL_FRAMES]
    for path in tqdm(paths, desc=f'{stiffness:g} kPa'):
        table = gel_scale_sweep(read_rgb(path), config, SIGMAS_UM, GEL_MPP)
        table['stiffness_kPa'] = stiffness
        table['filename'] = path.name
        gel_rows.append(table)
gel_sweep = pd.concat(gel_rows, ignore_index=True)

print(f"\nhistology: {histology_sweep.image_id.nunique()} images")
print(f"gels: {len(gel_rows)} frames across {len(usable)} stiffnesses")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for dx, group in histology_sweep.groupby('dx'):
    m = group.groupby('sigma_um').defect_density_mm2.median()
    axes[0].plot(m.index, m.values, marker='o', label=f'histology {dx}')
for stiffness, group in gel_sweep.groupby('stiffness_kPa'):
    m = group.groupby('sigma_um').defect_density_mm2.median()
    axes[0].plot(m.index, m.values, marker='s', ls='--',
                 label=f'gel {stiffness:g} kPa')
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('detection scale (um)')
axes[0].set_ylabel('defect density (mm$^{-2}$)')
axes[0].set_title('Raw density is scale-dependent')
axes[0].legend(fontsize=7)

for dx, group in histology_sweep.groupby('dx'):
    m = group.groupby('sigma_um').rho_xi2.median()
    axes[1].plot(m.index, m.values, marker='o', label=f'histology {dx}')
for stiffness, group in gel_sweep.groupby('stiffness_kPa'):
    m = group.groupby('sigma_um').rho_xi2.median()
    axes[1].plot(m.index, m.values, marker='s', ls='--',
                 label=f'gel {stiffness:g} kPa')
axes[1].set_xscale('log')
axes[1].set_xlabel('detection scale (um)')
axes[1].set_ylabel(r'$\rho\,\xi^2$  (dimensionless)')
axes[1].set_title('Normalised: the comparable quantity')
axes[1].legend(fontsize=7)

for dx, group in histology_sweep.groupby('dx'):
    m = group.groupby('sigma_um').global_S.median()
    axes[2].plot(m.index, m.values, marker='o', label=f'histology {dx}')
for stiffness, group in gel_sweep.groupby('stiffness_kPa'):
    m = group.groupby('sigma_um').global_S.median()
    axes[2].plot(m.index, m.values, marker='s', ls='--',
                 label=f'gel {stiffness:g} kPa')
axes[2].set_xscale('log')
axes[2].set_xlabel('detection scale (um)')
axes[2].set_ylabel('global nematic order S')
axes[2].set_title('Order')
axes[2].legend(fontsize=7)

plt.tight_layout(); plt.show()

print("If the normalised curves for the two systems collapse onto each other,")
print("they share a texture. If they stay separated at every scale, they do not,")
print("and no single-sigma comparison would have revealed that.")

## 5. Inference 1 — the sign of the stiffness response

The result worth reporting is the **sign** of the exponent.

A negative exponent means the elastic constant stiffens faster than activity, so
stiffer tissue is *more* ordered with *fewer* defects. That is the direction the
histology appears to show (UIP having higher `S` and lower nuclear defect
density than NSIP), and a fixed section could never establish it.

Three stiffnesses leave one degree of freedom after fitting two parameters, so
this is a description, not a test. Do not report an R-squared from it as
evidence.

In [ ]:
from lung_nematic.crossmap import calibration_curve, infer_stiffness

REFERENCE_SIGMA_UM = 28.0    # pick one where both systems detect reliably

gel_points = (gel_sweep[gel_sweep.sigma_um == REFERENCE_SIGMA_UM]
              .groupby('stiffness_kPa', as_index=False)
              .rho_xi2.median())
print(gel_points.round(4).to_string(index=False))

curve = calibration_curve(gel_points)
direction = ('DECREASES' if curve['log_slope'] < 0 else 'INCREASES')
print(f"\nrho*xi^2 {direction} with stiffness")
print(f"  exponent {curve['log_slope']:+.3f} over "
      f"{curve['stiffness_range_kPa'][0]:g}-{curve['stiffness_range_kPa'][1]:g} kPa "
      f"({curve['n_points']} points)")
print(f"\n  negative -> elasticity outpaces activity: stiffer tissue is more")
print(f"              ordered with fewer defects")
print(f"  positive -> activity outpaces elasticity: stiffer tissue is more")
print(f"              defective")

## 6. Inference 2 — effective stiffness of each histological region

Inverting the calibration turns a dimensionless density measured on a slide into
a stiffness estimate.

Values needing extrapolation beyond the calibrated range are flagged. A power
law fitted on 5–23 kPa says nothing about 200 kPa, and an unflagged number there
would be read as a measurement.

In [ ]:
histology_at_ref = histology_sweep[
    histology_sweep.sigma_um == REFERENCE_SIGMA_UM
].dropna(subset=['rho_xi2'])

inferred = infer_stiffness(histology_at_ref.rho_xi2.to_numpy(), curve)
inferred['image_id'] = histology_at_ref.image_id.to_numpy()
inferred['dx'] = histology_at_ref.dx.to_numpy()
inferred['patient'] = histology_at_ref.patient.to_numpy()

print("per patient (median of images), UIP vs NSIP:\n")
per_patient = (inferred[inferred.within_calibrated_range]
               .groupby(['dx', 'patient'])
               .inferred_stiffness_kPa.median())
print(per_patient.round(1).to_string())

outside = (~inferred.within_calibrated_range).sum()
print(f"\n{outside}/{len(inferred)} images fell outside the calibrated range "
      f"and are excluded above")

print("\nVALIDATE THIS. Predict E from texture on one section, measure E by")
print("instrumented microindentation on the adjacent section, compare. Without")
print("that loop these numbers are an extrapolation from fibroblasts on plastic")
print("gels to myofibroblasts in a three-dimensional matrix.")

## 7. Inference 3 — is the lesion equilibrated?

Fibroblastic foci are held to be the active leading edge of the disease. If
their texture sits **above** the steady-state band of every gel stiffness, they
have not finished coarsening — mechanistic support for that hypothesis, read
from a fixed section.

In [ ]:
band_low = gel_points.rho_xi2.min()
band_high = gel_points.rho_xi2.max()
print(f"gel steady-state band at {REFERENCE_SIGMA_UM:g} um: "
      f"{band_low:.4f} to {band_high:.4f}\n")

for dx, group in histology_at_ref.groupby('dx'):
    above = (group.rho_xi2 > band_high).mean()
    below = (group.rho_xi2 < band_low).mean()
    print(f"{dx:5s}  median {group.rho_xi2.median():.4f}  |  "
          f"{above:.0%} above band, {below:.0%} below, "
          f"{1 - above - below:.0%} within")

print("\nabove the band -> still coarsening, not equilibrated")
print("within         -> consistent with a mature, steady lesion")
print("below          -> more ordered than any gel reaches; check the")
print("                  sectioning diagnostic in the next cell first")

## 8. The sectioning diagnostic — read this before believing anything above

A gel is a genuine two-dimensional nematic where defects are points created and
destroyed in pairs, so the two charge classes balance. A histological section
cuts a **three-dimensional** disclination network, where defects are lines; the
apparent charge depends on the cutting angle and the classes need not balance.

An imbalance larger than anything the gels show at any stiffness means the
section is not a clean two-dimensional slice, and its density is not directly
comparable to a gel's. In that case the inferred stiffnesses above are not
trustworthy and the honest move is to stratify by section geometry — which needs
the focus domain segmented.

In [ ]:
hist_summary = pd.read_csv(
    Path('/scratch/histology/resultados_v2/summary_metrics_nuclear.csv'))
hist_summary['imbalance'] = (
    (hist_summary.n_plus_half - hist_summary.n_minus_half).abs()
    / (hist_summary.n_plus_half + hist_summary.n_minus_half).replace(0, np.nan)
)

tracks = pd.read_csv(TIMELAPSE_RESULTS / 'defect_tracks.tsv', sep='\t')
gel_imbalance = []
for (stiffness, frame), group in tracks.groupby(['stiffness_kPa', 'frame']):
    plus = (group.charge > 0).sum(); minus = (group.charge < 0).sum()
    if plus + minus:
        gel_imbalance.append(abs(plus - minus) / (plus + minus))
gel_imbalance = np.array(gel_imbalance)

print(f"gel charge imbalance:       median {np.median(gel_imbalance):.3f}  "
      f"95th pct {np.percentile(gel_imbalance, 95):.3f}")
print(f"histology charge imbalance: median "
      f"{hist_summary.imbalance.median():.3f}  "
      f"95th pct {hist_summary.imbalance.quantile(0.95):.3f}")

exceeds = (hist_summary.imbalance
           > np.percentile(gel_imbalance, 95)).mean()
print(f"\n{exceeds:.0%} of histology images exceed the gel 95th percentile")
if exceeds > 0.3:
    print("! most sections are not clean 2D slices. Treat the inferred")
    print("  stiffnesses as provisional and stratify by section geometry.")

## 9. Save

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
histology_sweep.to_csv(OUTPUT_DIR / 'histology_scale_sweep.tsv', sep='\t', index=False)
gel_sweep.to_csv(OUTPUT_DIR / 'gel_scale_sweep.tsv', sep='\t', index=False)
inferred.to_csv(OUTPUT_DIR / 'inferred_stiffness.tsv', sep='\t', index=False)
pd.DataFrame([curve]).to_csv(OUTPUT_DIR / 'calibration_curve.tsv', sep='\t', index=False)
print(f"written to {OUTPUT_DIR}")

## Reading the result

The chain is only as strong as its weakest link, and the links are checkable in
order:

1. **Fewer than two steady stiffnesses** (cell 3) — stop. No calibration, and
   any stiffness below is fabricated.
2. **Normalised curves do not collapse** (cell 4) — the two systems do not share
   a texture. That is itself a finding, and it means the inversion is mapping
   between different physics.
3. **Charge imbalance exceeds the gels** (cell 8) — the sections are cutting a
   3D network. Stiffness estimates are provisional.
4. **All three pass** — the inferred stiffnesses are worth taking to
   microindentation, which is the only thing that can confirm them.

One statistical caution carried over from the histology analysis: images are
clustered within patients (ICC 0.14–0.29), so image-level spread understates
uncertainty. Aggregate to one value per patient before comparing UIP to NSIP,
and remember the two NSIP patients cannot support a group comparison whatever
the p-value says.